## Parte 1 — Softmax con temperatura

La softmax con temperatura se implementa en NumPy de forma numéricamente estable restando el máximo de los logits antes de exponenciar: con m = max_j z_j se calcula p_i = exp((z_i − m)/T) / Σ_j exp((z_j − m)/T), algebraicamente idéntica a p_i = exp(z_i/T) / Σ_j exp(z_j/T) pero sin riesgo de desbordamiento. Para los logits del enunciado (t0 = 2.0, t1 = 1.0, t2 = 0.5, t3 = 0.2, t4 = −1.0, t5 = −3.0) el máximo es m = 2.0. La entropía se calcula en bits como H = −Σ_i p_i log2(p_i). La celda siguiente calcula las tres distribuciones y sus entropías y genera la tabla y el gráfico de barras (`figuras/T1_tabla_softmax_temperatura.png`, `figuras/T1_softmax_temperatura_barras.png`).

| Token | T = 0.5 | T = 1.0 | T = 2.0 |
|---|---:|---:|---:|
| t0 | 0.8231 | 0.5516 | 0.3583 |
| t1 | 0.1114 | 0.2029 | 0.2173 |
| t2 | 0.0410 | 0.1231 | 0.1693 |
| t3 | 0.0225 | 0.0912 | 0.1457 |
| t4 | 0.0020 | 0.0275 | 0.0800 |
| t5 | 0.0000 | 0.0037 | 0.0294 |
| **Entropía (bits)** | **0.9147** | **1.7998** | **2.2888** |

Las tres distribuciones suman exactamente 1.0. El efecto de la temperatura es claro: al pasar de T = 0.5 a T = 2 la masa se reparte desde el token dominante t0 (0.8231 → 0.3583) hacia el resto (t5 sube de 3.7367e-05, que a cuatro decimales se muestra como 0.0000, hasta 0.0294), y la entropía crece de forma monótona: 0.9147 → 1.7998 → 2.2888 bits.

In [1]:
# T1: código aprobado por el crítico
# Subtarea T1: Softmax con temperatura (NumPy, estable) + entropía en bits
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ---------- Funciones ----------
def softmax_con_temperatura(logits, T):
    """p_i = exp(z_i/T) / sum_j exp(z_j/T), restando el máximo para estabilidad."""
    z = np.asarray(logits, dtype=np.float64)
    x = z / float(T)
    x = x - np.max(x)          # truco de estabilidad numérica
    e = np.exp(x)
    return e / np.sum(e)


def entropia_bits(p):
    """H(p) = -sum p_i log2 p_i, en bits."""
    p = np.asarray(p, dtype=np.float64)
    mask = p > 0
    return float(-np.sum(p[mask] * np.log2(p[mask])))


# ---------- Datos del enunciado ----------
logits = [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]
tokens = [f"t{i}" for i in range(6)]
temperaturas = [0.5, 1.0, 2.0]

# ---------- Cálculos ----------
distribuciones = {T: softmax_con_temperatura(logits, T) for T in temperaturas}
entropias = {T: entropia_bits(distribuciones[T]) for T in temperaturas}
sumas = {T: float(np.sum(distribuciones[T])) for T in temperaturas}

# ---------- Tabla con cuatro decimales ----------
tabla = pd.DataFrame(
    {f"T={T}": distribuciones[T] for T in temperaturas},
    index=tokens,
)
tabla.loc["Entropía (bits)"] = [entropias[T] for T in temperaturas]
tabla_fmt = tabla.apply(lambda col: col.map(lambda v: f"{v:.4f}"))

# ---------- Guardar tabla como CSV (apoyo) ----------
tabla_fmt.to_csv("tabla_softmax_temperatura.csv")

# ---------- Figura 1: tabla ----------
col_labels = ["Token"] + [f"T={T}" for T in temperaturas]
cell_text = [[tokens[i]] + [f"{distribuciones[T][i]:.4f}" for T in temperaturas]
             for i in range(6)]
cell_text.append(["Entropía (bits)"] + [f"{entropias[T]:.4f}" for T in temperaturas])

fig, ax = plt.subplots(figsize=(7.0, 3.4))
ax.axis("off")
tbl = ax.table(cellText=cell_text, colLabels=col_labels, loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.0, 1.5)
ax.set_title("Softmax con temperatura: distribuciones y entropía (bits)", pad=12)
plt.savefig("tabla_softmax_temperatura.png", dpi=120, bbox_inches="tight")
plt.close(fig)

# ---------- Figura 2: barras de las tres distribuciones ----------
fig2, ax2 = plt.subplots(figsize=(7.5, 4.2))
x = np.arange(len(tokens))
width = 0.25
for k, T in enumerate(temperaturas):
    ax2.bar(x + (k - 1) * width, distribuciones[T], width, label=f"T={T}")
ax2.set_xticks(x)
ax2.set_xticklabels(tokens)
ax2.set_ylabel("Probabilidad")
ax2.set_xlabel("Token")
ax2.set_title("Distribución softmax para T = 0.5, 1 y 2")
ax2.legend()
plt.savefig("softmax_temperatura_barras.png", dpi=120, bbox_inches="tight")
plt.close(fig2)

# ---------- resultados.json (contrato) ----------
resultados = {
    "subtarea": "T1_softmax_con_temperatura",
    "logits": logits,
    "tokens": tokens,
    "temperaturas": temperaturas,
    "distribuciones": {
        f"T={T}": [float(p) for p in distribuciones[T]] for T in temperaturas
    },
    "entropias_bits": {f"T={T}": entropias[T] for T in temperaturas},
    "suma_distribuciones": {f"T={T}": sumas[T] for T in temperaturas},
    "tabla_4_decimales": tabla_fmt.to_dict(orient="index"),
    "figuras": [
        "tabla_softmax_temperatura.png",
        "softmax_temperatura_barras.png",
        "tabla_softmax_temperatura.csv",
    ],
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ---------- Resumen ----------
print("=== T1: Softmax con temperatura ===")
print(f"Logits: {logits}")
print(tabla_fmt.to_string())
print("\nSumas de cada distribución:")
for T in temperaturas:
    print(f"  T={T}: suma = {sumas[T]:.10f}")
print("\nEntropías (bits):")
for T in temperaturas:
    print(f"  T={T}: H = {entropias[T]:.4f} bits")
print("\nArchivos generados: resultados.json, tabla_softmax_temperatura.png, "
      "softmax_temperatura_barras.png, tabla_softmax_temperatura.csv")


=== T1: Softmax con temperatura ===
Logits: [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]
                  T=0.5   T=1.0   T=2.0
t0               0.8231  0.5516  0.3583
t1               0.1114  0.2029  0.2173
t2               0.0410  0.1231  0.1693
t3               0.0225  0.0912  0.1457
t4               0.0020  0.0275  0.0800
t5               0.0000  0.0037  0.0294
Entropía (bits)  0.9147  1.7998  2.2888

Sumas de cada distribución:
  T=0.5: suma = 1.0000000000
  T=1.0: suma = 1.0000000000
  T=2.0: suma = 1.0000000000

Entropías (bits):
  T=0.5: H = 0.9147 bits
  T=1.0: H = 1.7998 bits
  T=2.0: H = 2.2888 bits

Archivos generados: resultados.json, tabla_softmax_temperatura.png, softmax_temperatura_barras.png, tabla_softmax_temperatura.csv


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T = 1 de la Parte 1 se ordenan los tokens de mayor a menor probabilidad (el orden queda t0, t1, t2, t3, t4, t5) y se acumula hasta alcanzar p = 0.9. Tras t2 la acumulada es 0.8776, todavía por debajo del umbral; al añadir t3 llega a 0.9688 ≥ 0.9, de modo que el núcleo mínimo es {t0, t1, t2, t3}. Quedan descartados t4 y t5, cuya masa conjunta es 0.0312. La distribución renormalizada se obtiene dividiendo las probabilidades del núcleo entre su masa acumulada (0.9688) y suma 1.0. La celda siguiente genera la tabla y la figura (`figuras/T2_top_p_nucleo_T1.png`).

| Token | p(T=1) | Acumulada | ¿Sobrevive? | Renormalizada |
|---|---:|---:|:---:|---:|
| t0 | 0.5516 | 0.5516 | sí | 0.5694 |
| t1 | 0.2029 | 0.7546 | sí | 0.2095 |
| t2 | 0.1231 | 0.8776 | sí | 0.1270 |
| t3 | 0.0912 | 0.9688 | sí | 0.0941 |
| t4 | 0.0275 | 0.9963 | no | — |
| t5 | 0.0037 | 1.0000 | no | — |

Sobreviven t0, t1, t2 y t3. Como ningún token individual alcanza por sí solo el umbral, el núcleo acumula 0.9688, algo por encima de 0.9, y el recorte solo elimina 0.0312 de la masa. La renormalización eleva ligeramente las probabilidades supervivientes: t0 pasa de 0.5516 a 0.5694 y t3 de 0.0912 a 0.0941.

In [2]:
# T2: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
Subtarea T2: Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T = 1
obtenida en la subtarea T1 (softmax con temperatura).

Pasos:
  1) Cargar la distribución de T = 1.0 desde entrada/T1/resultados.json.
  2) Ordenar los tokens por probabilidad descendente y acumular.
  3) Conservar el conjunto MÍNIMO de tokens cuya probabilidad acumulada alcanza p = 0.9.
  4) Renormalizar las probabilidades del núcleo para que sumen 1.0.
  5) Escribir resultados.json (contrato), una figura PNG y una tabla CSV.
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

P = 0.9  # umbral del núcleo (top-p)

# ----------------------------------------------------------------------
# 1) Cargar la distribución de T = 1.0 producida por la subtarea T1
# ----------------------------------------------------------------------
ruta_t1 = Path("entrada") / "T1" / "resultados.json"
if ruta_t1.exists():
    with open(ruta_t1, "r", encoding="utf-8") as f:
        t1 = json.load(f)
    tokens = list(t1["tokens"])
    dists = t1["distribuciones"]
    if "T=1.0" in dists:
        probs_T1 = np.asarray(dists["T=1.0"], dtype=float)
    else:  # respaldo: localizar la clave de temperatura 1.0
        clave = next(k for k in dists if abs(float(str(k).split("=")[1]) - 1.0) < 1e-12)
        probs_T1 = np.asarray(dists[clave], dtype=float)
    fuente = "distribucion T=1.0 de entrada/T1/resultados.json"
else:
    # Respaldo: recalcular la softmax con T = 1 a partir de los logits de la Parte 1
    tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    z = logits - logits.max()          # estabilidad numérica
    e = np.exp(z)
    probs_T1 = e / e.sum()
    fuente = "softmax(T=1) recalculada desde los logits de la Parte 1"

# ----------------------------------------------------------------------
# 2) Top-p: ordenar descendente, acumular y conservar el conjunto mínimo
#    cuya probabilidad acumulada alcanza p = 0.9
# ----------------------------------------------------------------------
orden = np.argsort(-probs_T1, kind="stable")            # índices en orden descendente
tokens_ord = [tokens[i] for i in orden]
probs_ord = probs_T1[orden]
acumulada = np.cumsum(probs_ord)

k = 0
acum = 0.0
for pr in probs_ord:                                    # conjunto mínimo con acumulada >= p
    k += 1
    acum += float(pr)
    if acum >= P:
        break
masa_nucleo = acum                                      # probabilidad acumulada del núcleo
masa_descartada = 1.0 - masa_nucleo

idx_nucleo = orden[:k]                                  # índices que sobreviven
tokens_sobrev = [tokens[i] for i in idx_nucleo]
tokens_fuera = [tokens[i] for i in orden[k:]]

# ----------------------------------------------------------------------
# 3) Renormalizar la distribución del núcleo
# ----------------------------------------------------------------------
probs_nucleo = probs_T1[idx_nucleo]
probs_renorm = probs_nucleo / probs_nucleo.sum()
dist_renorm = {tokens[i]: float(q) for i, q in zip(idx_nucleo, probs_renorm)}
suma_renorm = float(probs_renorm.sum())

# ----------------------------------------------------------------------
# 4) Tabla resumen (valores completos y a 4 decimales)
# ----------------------------------------------------------------------
acum_por_token = {}
ac_aux = 0.0
for tk, pr in zip(tokens_ord, probs_ord):
    ac_aux += float(pr)
    acum_por_token[tk] = ac_aux

tabla = pd.DataFrame({
    "token": tokens,
    "prob_T1": probs_T1,
    "prob_acumulada_desc": [acum_por_token[tk] for tk in tokens],
    "sobrevive_top_p": ["sí" if tk in tokens_sobrev else "no" for tk in tokens],
    "prob_renormalizada": [dist_renorm.get(tk, np.nan) for tk in tokens],
})

tabla_4dec = {
    tk: {
        "prob_T1": f"{probs_T1[i]:.4f}",
        "acumulada": f"{acum_por_token[tk]:.4f}",
        "sobrevive": "sí" if tk in tokens_sobrev else "no",
        "renormalizada": (f"{dist_renorm[tk]:.4f}" if tk in dist_renorm else "—"),
    }
    for i, tk in enumerate(tokens)
}

# ----------------------------------------------------------------------
# 5) Figura: distribución ordenada con frontera del núcleo + renormalizada
# ----------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))

colores = ["#2a9d8f" if tk in tokens_sobrev else "#e76f51" for tk in tokens_ord]
x = np.arange(len(tokens_ord))
axes[0].bar(x, probs_ord, color=colores, edgecolor="black")
axes[0].set_xticks(x)
axes[0].set_xticklabels(tokens_ord)
axes[0].set_ylabel("Probabilidad (T = 1)")
axes[0].set_title("Distribución T=1 ordenada (verde = núcleo top-p)")
for xi, pr in zip(x, probs_ord):
    axes[0].text(xi, pr + 0.008, f"{pr:.4f}", ha="center", fontsize=8)

ax2 = axes[0].twinx()
ax2.plot(x, acumulada, "o--", color="#264653", lw=1.5, label="Acumulada")
ax2.axhline(P, color="red", ls=":", lw=1.5, label=f"p = {P}")
ax2.set_ylabel("Probabilidad acumulada")
ax2.set_ylim(0, 1.05)
ax2.legend(loc="lower right", fontsize=8)

xr = np.arange(k)
axes[1].bar(xr, probs_renorm, color="#2a9d8f", edgecolor="black")
axes[1].set_xticks(xr)
axes[1].set_xticklabels(tokens_sobrev)
axes[1].set_ylabel("Probabilidad renormalizada")
axes[1].set_title(f"Distribución renormalizada (suma = {suma_renorm:.4f})")
for xi, q in zip(xr, probs_renorm):
    axes[1].text(xi, q + 0.008, f"{q:.4f}", ha="center", fontsize=8)
axes[1].set_ylim(0, float(probs_renorm.max()) * 1.18)

fig.suptitle(f"Muestreo de núcleo (top-p) con p = {P} sobre la distribución de T = 1", y=1.02)
fig.tight_layout()
fig.savefig("top_p_nucleo_T1.png", dpi=120, bbox_inches="tight")
plt.close(fig)

# ----------------------------------------------------------------------
# 6) Guardar tabla CSV y resultados.json (contrato de la subtarea)
# ----------------------------------------------------------------------
tabla.to_csv("tabla_top_p_T1.csv", index=False)

resultados = {
    "subtarea": "T2_top_p_nucleus_sampling",
    "p": P,
    "fuente": fuente,
    "tokens": tokens,
    "probabilidades_T1": probs_T1.tolist(),
    "orden_descendente": tokens_ord,
    "probabilidades_ordenadas": probs_ord.tolist(),
    "probabilidad_acumulada_ordenada": acumulada.tolist(),
    "tokens_sobrevivientes": tokens_sobrev,
    "tokens_descartados": tokens_fuera,
    "num_tokens_nucleo": k,
    "probabilidad_acumulada_nucleo": masa_nucleo,
    "masa_descartada": masa_descartada,
    "probabilidades_originales_nucleo": {tokens[i]: float(probs_T1[i]) for i in idx_nucleo},
    "distribucion_renormalizada": dist_renorm,
    "suma_distribucion_renormalizada": suma_renorm,
    "tabla_4_decimales": tabla_4dec,
    "figuras": ["top_p_nucleo_T1.png", "tabla_top_p_T1.csv"],
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------------------
# 7) Resumen por consola
# ----------------------------------------------------------------------
print("=== T2: Muestreo de núcleo (top-p), p = 0.9, sobre la distribución de T = 1 ===")
print("Fuente:", fuente)
print("Distribución T=1:", {tk: round(float(p), 6) for tk, p in zip(tokens, probs_T1)})
print("Orden descendente:", tokens_ord)
print("Acumulada:", [round(float(a), 6) for a in acumulada])
print(f"Tokens sobrevivientes (conjunto mínimo con acumulada >= {P}): {tokens_sobrev}")
print(f"Tokens descartados: {tokens_fuera}")
print(f"Probabilidad acumulada del núcleo: {masa_nucleo:.6f} (masa descartada: {masa_descartada:.6f})")
print("Distribución renormalizada:")
for tk in tokens_sobrev:
    print(f"  {tk}: {dist_renorm[tk]:.6f}")
print(f"Suma de la distribución renormalizada: {suma_renorm:.12f}")
print("Archivos escritos: resultados.json, top_p_nucleo_T1.png, tabla_top_p_T1.csv")


=== T2: Muestreo de núcleo (top-p), p = 0.9, sobre la distribución de T = 1 ===
Fuente: distribucion T=1.0 de entrada/T1/resultados.json
Distribución T=1: {'t0': 0.551623, 't1': 0.202931, 't2': 0.123084, 't3': 0.091183, 't4': 0.027464, 't5': 0.003717}
Orden descendente: ['t0', 't1', 't2', 't3', 't4', 't5']
Acumulada: [0.551623, 0.754553, 0.877637, 0.96882, 0.996283, 1.0]
Tokens sobrevivientes (conjunto mínimo con acumulada >= 0.9): ['t0', 't1', 't2', 't3']
Tokens descartados: ['t4', 't5']
Probabilidad acumulada del núcleo: 0.968820 (masa descartada: 0.031180)
Distribución renormalizada:
  t0: 0.569376
  t1: 0.209462
  t2: 0.127045
  t3: 0.094117
Suma de la distribución renormalizada: 1.000000000000
Archivos escritos: resultados.json, top_p_nucleo_T1.png, tabla_top_p_T1.csv


## Parte 3 — Comprobación empírica

Se generan 10 000 muestras de la distribución renormalizada de la Parte 2 con `numpy.random.default_rng(0)` y se comparan las frecuencias observadas con las probabilidades teóricas. La divergencia KL se calcula en bits como KL(q‖p) = Σ_i q_i log2(q_i / p_i), con q la distribución empírica y p la teórica. La celda siguiente realiza el muestreo, produce el gráfico de barras comparativo (`figuras/T3_barras_frecuencias_vs_teoricas.png`) y calcula la KL.

| Token | Teórica | Conteo observado | Frecuencia observada |
|---|---:|---:|---:|
| t0 | 0.5694 | 5682 | 0.5682 |
| t1 | 0.2095 | 2093 | 0.2093 |
| t2 | 0.1270 | 1296 | 0.1296 |
| t3 | 0.0941 | 929 | 0.0929 |

Las frecuencias observadas suman 1.0 y reproducen de cerca las teóricas: en la figura las barras de ambas series son casi indistinguibles y, comparando las columnas, las discrepancias solo se manifiestan a partir de la tercera cifra decimal. La divergencia KL de la empírica respecto de la teórica es 5.0069e-05 bits (0.0001 al redondear a cuatro decimales), un valor prácticamente nulo que confirma que el muestreo reproduce la distribución del núcleo y que las diferencias apreciables son únicamente el ruido esperable del muestreo con 10 000 muestras.

In [3]:
# T3: código aprobado por el crítico
import json
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------------------------------------------------------
# T3: Comprobación empírica del muestreo top-p (Parte 3)
# ---------------------------------------------------------------

# 1) Cargar la distribución renormalizada de la Parte 2 (T2)
ruta_t2 = Path("entrada/T2/resultados.json")
with open(ruta_t2, "r", encoding="utf-8") as f:
    t2 = json.load(f)

tokens = list(t2["tokens_sobrevivientes"])  # ["t0", "t1", "t2", "t3"]
p_teo = np.array([t2["distribucion_renormalizada"][t] for t in tokens], dtype=float)

# Verificación de sanidad de la distribución teórica
assert abs(p_teo.sum() - 1.0) < 1e-9, "La distribución renormalizada no suma 1"

# 2) Generar 10,000 muestras con la semilla exigida
N = 10000
rng = np.random.default_rng(0)
indices = np.arange(len(tokens))
muestras = rng.choice(indices, size=N, p=p_teo)

# 3) Frecuencias observadas
conteos = np.bincount(muestras, minlength=len(tokens))
p_emp = conteos / N

# 4) Divergencia KL de la empírica respecto de la teórica, en bits:
#    D_KL(P_emp || P_teo) = sum_i p_emp_i * log2(p_emp_i / p_teo_i)
# (todos los tokens del núcleo tienen p >= 0.09, así que no hay conteos nulos,
#  pero se protege el cálculo por robustez)
mask = p_emp > 0
kl_bits = float(np.sum(p_emp[mask] * np.log2(p_emp[mask] / p_teo[mask])))

# 5) Gráfico de barras: frecuencias observadas vs probabilidades teóricas
x = np.arange(len(tokens))
width = 0.38
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - width / 2, p_emp, width, label="Frecuencia observada (empírica)",
            color="#4C72B0", edgecolor="black")
b2 = ax.bar(x + width / 2, p_teo, width, label="Probabilidad teórica (renormalizada)",
            color="#DD8452", edgecolor="black")
for bars in (b1, b2):
    for b in bars:
        ax.annotate(f"{b.get_height():.4f}",
                    (b.get_x() + b.get_width() / 2, b.get_height()),
                    ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("Probabilidad / frecuencia relativa")
ax.set_ylim(0, max(p_teo.max(), p_emp.max()) * 1.18)
ax.set_title(f"Muestreo del núcleo top-p = 0.9 (T = 1): {N} muestras, default_rng(0)\n"
             f"KL(empírica || teórica) = {kl_bits:.6f} bits")
ax.legend()
fig.tight_layout()
fig.savefig("barras_frecuencias_vs_teoricas.png", dpi=120)
plt.close(fig)

# 6) Escribir resultados.json (contrato de la subtarea)
resultados = {
    "subtarea": "T3_comprobacion_empirica_top_p",
    "n_muestras": N,
    "semilla": 0,
    "rng": "numpy.random.default_rng(0)",
    "fuente": "distribucion_renormalizada de entrada/T2/resultados.json",
    "tokens": tokens,
    "probabilidades_teoricas": {t: float(p) for t, p in zip(tokens, p_teo)},
    "conteos_observados": {t: int(c) for t, c in zip(tokens, conteos)},
    "frecuencias_observadas": {t: float(f) for t, f in zip(tokens, p_emp)},
    "suma_frecuencias_observadas": float(p_emp.sum()),
    "divergencia_KL_bits": kl_bits,
    "divergencia_KL_bits_4dec": f"{kl_bits:.4f}",
    "figura": "barras_frecuencias_vs_teoricas.png",
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# 7) Resumen breve
print("T3 — Comprobación empírica del muestreo top-p (núcleo de la Parte 2)")
print(f"Muestras: {N} | Semilla: numpy.random.default_rng(0)")
print(f"{'token':<8}{'p_teorica':>14}{'conteo':>10}{'p_empirica':>14}")
for t, pt, c, pe in zip(tokens, p_teo, conteos, p_emp):
    print(f"{t:<8}{pt:>14.6f}{c:>10d}{pe:>14.6f}")
print(f"Divergencia KL (empírica || teórica) = {kl_bits:.6f} bits")
print("Figura guardada: barras_frecuencias_vs_teoricas.png")
print("Resultados escritos en resultados.json")


T3 — Comprobación empírica del muestreo top-p (núcleo de la Parte 2)
Muestras: 10000 | Semilla: numpy.random.default_rng(0)
token        p_teorica    conteo    p_empirica
t0            0.569376      5682      0.568200
t1            0.209462      2093      0.209300
t2            0.127045      1296      0.129600
t3            0.094117       929      0.092900
Divergencia KL (empírica || teórica) = 0.000050 bits
Figura guardada: barras_frecuencias_vs_teoricas.png
Resultados escritos en resultados.json


## Parte 4 — Pregunta conceptual

**Cuando T → 0.** En p_i ∝ exp(z_i/T), dividir los logits por una temperatura cada vez más pequeña amplifica sin cota las diferencias entre ellos: exp((z_i − m)/T) tiende a 1 para el token de máximo logit (t0, con z = 2.0) y a 0 para todos los demás, cuyo exponente tiende a menos infinito. La distribución converge así a una masa puntual sobre t0: p(t0) → 1 y p(t_i) → 0 para el resto. La decodificación voraz (greedy) elige exactamente argmax_i z_i, que aquí es t0; como el argmax de la softmax coincide con el de los logits para cualquier T, en el límite T → 0 toda la masa queda sobre ese token, de modo que tanto muestrear la distribución como tomar su modo reproduce la elección voraz con certeza. Por eso T → 0 equivale a greedy.

La evidencia de la Parte 1 muestra esa tendencia: al bajar T de 2 a 0.5, p(t0) sube 0.3583 → 0.5516 → 0.8231, p(t5) cae 0.0294 → 0.0037 → 3.7367e-05, y la entropía desciende monótonamente 2.2888 → 1.7998 → 0.9147 bits. Extrapolando la secuencia, en T → 0 la entropía tiende a 0 bits, el mínimo posible, correspondiente a una distribución determinista concentrada en un solo token.

**Cuando T → ∞.** Dividir los logits por una temperatura enorme hace z_i/T → 0 para todos los tokens, de modo que todos los exponenciales valen 1 y p_i → 1/6 para cada uno de los seis tokens: la distribución tiende a la uniforme y su entropía al máximo posible para este vocabulario, log2(6) bits. Los datos ya apuntan en esa dirección: con T = 2 la distribución se aplana (t0 baja a 0.3583 y t5 sube a 0.0294) y la entropía alcanza 2.2888 bits, el mayor de los tres valores medidos, acercándose al máximo uniforme.

En síntesis, la temperatura interpola entre los dos extremos que las entropías medidas trazan de forma monótona —0.9147, 1.7998 y 2.2888 bits para T = 0.5, 1 y 2—: T → 0 concentra toda la masa en el token de mayor logit (entropía → 0 bits, decodificación voraz determinista) y T → ∞ la reparte por igual entre los seis tokens (entropía → log2(6) bits, sorteo prácticamente uniforme).